# Correlation Analysis with Python

This notebook introduces **Pearson correlation** using a small, reproducible teaching dataset generated inside the notebook.

It does not require Google Drive, external files, or downloaded datasets.

## Learning objectives

After completing the lesson, students should be able to:

1. Explain the direction and strength of a linear correlation.
2. Calculate Pearson's correlation coefficient and p-value.
3. Interpret a correlation matrix.
4. Visualize relationships using scatter plots.
5. Distinguish statistical significance from practical importance.
6. Explain why correlation does not establish causation.

## 1. Import the required libraries

The notebook uses libraries already listed in the repository's common `requirements.txt` file.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

## 2. Create a reproducible teaching dataset

The following variables are synthetic:

- `study_hours`: weekly study time;
- `exam_score`: assessment result influenced partly by study time;
- `screen_time`: daily non-study screen time;
- `sleep_hours`: average nightly sleep;
- `student_id`: an identifier with no analytical meaning.

The generated relationships are deliberately simplified for teaching. They should not be interpreted as evidence about real students.

In [ ]:
sample_size = 120

study_hours = rng.uniform(1, 20, sample_size)
screen_time = rng.uniform(1, 9, sample_size)
sleep_hours = rng.normal(7, 0.9, sample_size)

exam_score = (
    48
    + 2.1 * study_hours
    - 1.4 * screen_time
    + 1.2 * sleep_hours
    + rng.normal(0, 7, sample_size)
)

exam_score = np.clip(exam_score, 0, 100)

data = pd.DataFrame(
    {
        "student_id": np.arange(1, sample_size + 1),
        "study_hours": study_hours,
        "screen_time": screen_time,
        "sleep_hours": sleep_hours,
        "exam_score": exam_score,
    }
)

print("Dataset shape:", data.shape)
data.head()

## 3. Inspect and summarize the data

Before calculating correlations, inspect the schema, missing values, and descriptive statistics.

In [ ]:
print("Column data types:")
print(data.dtypes)

print("\nMissing values:")
print(data.isna().sum())

data.describe().round(2)

## 4. Calculate one Pearson correlation

Pearson's correlation coefficient, usually written as **r**, measures the strength and direction of a linear relationship.

- `r` close to `+1`: strong positive linear relationship;
- `r` close to `-1`: strong negative linear relationship;
- `r` close to `0`: weak or absent linear relationship.

The p-value tests the null hypothesis that the population correlation is zero. A small p-value does not automatically mean the relationship is large or practically important.

In [ ]:
correlation, p_value = pearsonr(
    data["study_hours"],
    data["exam_score"],
)

print(f"Pearson r: {correlation:.3f}")
print(f"p-value: {p_value:.6g}")

## 5. Interpret correlation strength

The following helper provides a simple classroom interpretation. These labels are conventions, not universal scientific thresholds.

In [ ]:
def describe_correlation(value: float) -> str:
    magnitude = abs(value)

    if magnitude < 0.20:
        strength = "very weak"
    elif magnitude < 0.40:
        strength = "weak"
    elif magnitude < 0.60:
        strength = "moderate"
    elif magnitude < 0.80:
        strength = "strong"
    else:
        strength = "very strong"

    if value > 0:
        direction = "positive"
    elif value < 0:
        direction = "negative"
    else:
        direction = "no"

    return f"{strength} {direction} linear relationship"


print(describe_correlation(correlation))

## 6. Visualize a relationship

A scatter plot helps reveal whether the relationship is linear and whether unusual observations are present.

In [ ]:
x = data["study_hours"].to_numpy()
y = data["exam_score"].to_numpy()

slope, intercept = np.polyfit(x, y, 1)
fitted_values = slope * x + intercept

plt.figure(figsize=(8, 5))
plt.scatter(x, y, alpha=0.7, label="Students")
plt.plot(x, fitted_values, linewidth=2, label="Linear trend")
plt.xlabel("Study hours per week")
plt.ylabel("Exam score")
plt.title("Study Hours and Exam Score")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Build a correlation matrix

Identifiers such as `student_id` should be excluded because an arbitrary record number is not a meaningful analytical feature.

In [ ]:
analysis_columns = [
    "study_hours",
    "screen_time",
    "sleep_hours",
    "exam_score",
]

correlation_matrix = (
    data[analysis_columns]
    .corr(method="pearson")
    .round(3)
)

correlation_matrix

## 8. Visualize the correlation matrix

The matrix is displayed with Matplotlib so the notebook does not require an additional plotting package.

In [ ]:
matrix_values = correlation_matrix.to_numpy()

fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(
    matrix_values,
    vmin=-1,
    vmax=1,
)

ax.set_xticks(range(len(analysis_columns)))
ax.set_yticks(range(len(analysis_columns)))
ax.set_xticklabels(analysis_columns, rotation=45, ha="right")
ax.set_yticklabels(analysis_columns)

for row in range(len(analysis_columns)):
    for column in range(len(analysis_columns)):
        ax.text(
            column,
            row,
            f"{matrix_values[row, column]:.2f}",
            ha="center",
            va="center",
        )

fig.colorbar(image, ax=ax, label="Pearson correlation")
ax.set_title("Correlation Matrix")
fig.tight_layout()
plt.show()

## 9. Calculate correlations with the target variable

This table provides the coefficient, p-value, and a simple verbal interpretation for each predictor.

In [ ]:
results = []

for feature in ["study_hours", "screen_time", "sleep_hours"]:
    coefficient, feature_p_value = pearsonr(
        data[feature],
        data["exam_score"],
    )

    results.append(
        {
            "feature": feature,
            "pearson_r": coefficient,
            "p_value": feature_p_value,
            "interpretation": describe_correlation(coefficient),
        }
    )

correlation_results = pd.DataFrame(results)
correlation_results.round(
    {
        "pearson_r": 3,
        "p_value": 6,
    }
)

## 10. Compare positive, negative, and weak relationships

Separate plots make the direction and strength easier to inspect.

In [ ]:
relationships = [
    ("study_hours", "Positive relationship"),
    ("screen_time", "Negative relationship"),
    ("sleep_hours", "Weaker relationship"),
]

for feature, subtitle in relationships:
    coefficient, _ = pearsonr(
        data[feature],
        data["exam_score"],
    )

    plt.figure(figsize=(7, 4))
    plt.scatter(
        data[feature],
        data["exam_score"],
        alpha=0.7,
    )
    plt.xlabel(feature.replace("_", " ").title())
    plt.ylabel("Exam Score")
    plt.title(f"{subtitle}: r = {coefficient:.3f}")
    plt.tight_layout()
    plt.show()

## 11. Correlation does not imply causation

A correlation can arise because:

- one variable influences another;
- the direction of influence is reversed;
- a third variable affects both;
- the sample is biased;
- the relationship occurs by chance;
- both variables change over time;
- the relationship is created by the way the data were collected.

For example, observing that study time and exam scores are correlated does not prove that increasing study time alone will produce the same score change for every learner. Prior knowledge, teaching quality, health, motivation, assessment design, and many other variables may contribute.

## Limitations of Pearson correlation

Pearson correlation:

- measures linear association;
- can be influenced by outliers;
- may miss strong nonlinear relationships;
- does not establish causation;
- should be interpreted with the sample size and context;
- requires careful handling of missing or invalid values.

## Exercises

1. Change `RANDOM_SEED` and observe how the coefficients change.
2. Increase the random noise added to `exam_score`.
3. Add an outlier and compare the correlation before and after.
4. Create a nonlinear relationship and calculate Pearson's r.
5. Compare Pearson correlation with Spearman rank correlation.
6. Explain why `student_id` should not be included in the analysis.
7. Replace the synthetic data with a properly documented local dataset.